# Fine-tune Gemma-3-270M for Entity Extraction on Hebrew Song Titles

This notebook fine-tunes the Gemma-3-270M model (quantized to 4-bit) for extracting entities (singers, songs, albums, genres, misc) from Hebrew song titles.

**Key Features:**
- Uses Unsloth for efficient fine-tuning with 4-bit quantization (QLoRA).
- Converts NER dataset to generative format.
- Dataset: NHLOCAL/SingNER (Hebrew song titles).
- Runs on Kaggle Notebooks with T4 x2 or P100.
- Final output: LoRA adapter for entity extraction.

## 📦 Setup and Installation

In [ ]:
%%capture
import os

# Install Unsloth and dependencies
!pip install unsloth
!pip install --no-deps bitsandbytes accelerate xformers peft trl transformers datasets huggingface_hub hf_transfer
!pip install sentencepiece protobuf
!pip install -q "transformers>=4.36.0"
!pip install -q "datasets>=2.14.0"
!pip install -q tqdm

In [ ]:
import torch
import numpy as np
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import TrainingArguments, AutoTokenizer
from trl import SFTTrainer
from datasets import load_dataset
from tqdm.auto import tqdm
import json
import time
import random
import os
import gc
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"🦥 Unsloth: Fast fine-tuning enabled")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_info = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_info.name}")
    print(f"GPU Memory: {gpu_info.total_memory / 1e9:.1f} GB")
    print(f"BFloat16 Support: {is_bfloat16_supported()}")

In [ ]:
def check_memory_usage():
    """Check and display current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        free = total - allocated
        
        print(f"\n💾 GPU Memory Status:")
        print(f"   Total: {total:.2f} GB")
        print(f"   Allocated: {allocated:.2f} GB")
        print(f"   Reserved: {reserved:.2f} GB")
        print(f"   Free: {free:.2f} GB")
        
        return allocated, free
    return 0, 0

# Run initial check
print("🔧 Memory management functions loaded")
check_memory_usage()

## ⚙️ Configuration

In [ ]:
CONFIG = {
    # Model settings
    "model_name": "google/gemma-3-270m",
    "cache_dir": "./models",
    
    # Context length
    "max_seq_length": 8192,
    
    # LoRA configuration
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    
    # Training settings
    "batch_size": 8,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "num_epochs": 3,
    "warmup_steps": 10,
    "weight_decay": 0.01,
    
    # Unsloth optimizations
    "use_gradient_checkpointing": "unsloth",
    "use_rslora": True,
    "load_in_4bit": True,
    "dtype": None,
}

print("📊 Fine-Tuning Configuration:")
print(f"   Base model: {CONFIG['model_name']}")
print(f"   Max sequence length: {CONFIG['max_seq_length']} tokens")
print(f"   LoRA rank: {CONFIG['lora_r']}")

## 🔐 HuggingFace Authentication

In [ ]:
from huggingface_hub import login

login()

print("✅ Authentication setup complete!")

## 🤖 Model Loading

In [ ]:
print("🚀 Loading model with Unsloth optimizations...")
print(f"   Target sequence length: {CONFIG['max_seq_length']} tokens")

check_memory_usage()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=CONFIG["dtype"],
    load_in_4bit=CONFIG["load_in_4bit"],
    cache_dir=CONFIG["cache_dir"],
)

print("\n✅ Model loaded with Unsloth!")
print(f"   Max sequence length: {CONFIG['max_seq_length']} tokens")
print(f"   Using dtype: {model.dtype}")

check_memory_usage()

print("\n🔧 Adding LoRA adapter...")
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=CONFIG["lora_target_modules"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing=CONFIG["use_gradient_checkpointing"],
    random_state=42,
    use_rslora=CONFIG["use_rslora"],
    loftq_config=None,
)

print("\n📊 Model Statistics:")
model.print_trainable_parameters()

print("✅ LoRA adapter configured!")

check_memory_usage()

## 📊 Load and Prepare Dataset

In [ ]:
print("📥 Loading dataset from HuggingFace...")
dataset = load_dataset("NHLOCAL/SingNER")

print("\n✅ Dataset loaded!")
print(f"   Train size: {len(dataset['train'])}")
print(f"   Test size: {len(dataset['test'])}")

In [ ]:
eos_token = tokenizer.eos_token

def format_example(example):
    text = example['text']
    entities = example['entities']
    
    grouped = defaultdict(list)
    for ent in sorted(entities, key=lambda x: x['start']):
        entity_text = text[ent['start']:ent['end']]
        grouped[ent['label']].append(entity_text)
    
    output = ""
    for label in ['SINGER', 'SONG', 'ALBUM', 'GENRE', 'MISC']:
        if grouped[label]:
            output += f"{label}s: {', '.join(grouped[label])}\n"
    output = output.strip()
    
    user_content = (
        "Extract the following entities from this Hebrew song title: "
        "singers, songs, albums, genres, misc.\n"
        f"Title: {text}"
    )
    
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]
    
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False) + eos_token
    
    return {"text": formatted_text}

print("\n🔄 Formatting dataset...")
formatted_dataset = dataset.map(format_example, num_proc=4)

print("\n✅ Dataset formatted!")
print("Example:")
print(formatted_dataset['train'][0]['text'])

## 🏋️ Train the Model

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    warmup_steps=CONFIG["warmup_steps"],
    num_train_epochs=CONFIG["num_epochs"],
    learning_rate=CONFIG["learning_rate"],
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=CONFIG["weight_decay"],
    lr_scheduler_type="linear",
    seed=42,
    output_dir="outputs",
    evaluation_strategy="steps",
    eval_steps=100,
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_num_proc=4,
    packing=False,
    args=training_args,
)

print("\n🚀 Starting training...")
trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(trainer_stats)

## 💾 Save Model and Push to HuggingFace Hub

In [ ]:
print("\n💾 Saving LoRA Adapter...")

adapter_name = "gemma-3-270m-hebrew-song-entity-extractor"
save_path = "./models/gemma_entity_extractor"

try:
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print("✅ Model saved successfully!")
    
    from huggingface_hub import HfApi
    api = HfApi()
    
    api.create_repo(repo_id=adapter_name, exist_ok=True)
    api.upload_folder(
        folder_path=save_path,
        repo_id=adapter_name,
        repo_type="model"
    )
    
    print(f"\n🎉 Successfully pushed to: https://huggingface.co/{adapter_name}")
except Exception as e:
    print(f"⚠️ Error: {e}")

print("\n🎊 Fine-Tuning Complete!")